# 2. Historical Replay Execution

This notebook executes the non-compensatory governance framework against three analytical layers: the 12 original expert-triangulated cases, the 20-case core-equivalent cohort (with EEE overlay), and the full 91-case expanded benchmark. Each layer uses a distinct feature-routing path, reflecting the tiered evidence architecture.

In [1]:
# Configuration
import json, sys, csv, os, copy
from pathlib import Path

for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "config" / "harness_settings.json").is_file():
        _s = str(_cand)
        if _s not in sys.path:
            sys.path.insert(0, _s)
        break
from p4_replay.bootstrap import prepare_notebook

REPO_ROOT = prepare_notebook(engine_on_path=True)
import corrected_public_engine_v1_1 as engine
from eee_overlay_adapter import merge_features

CANONICAL_PATH = REPO_ROOT / 'data' / 'canonical' / 'canonical_dataset.json'
BENCHMARK_DIR = REPO_ROOT / 'data' / 'benchmark' / 'cases'
EEE_PATH = REPO_ROOT / 'data' / 'benchmark' / 'eee_overlay' / 'canonicalised_v1.json'
CORE_EQUIV_PATH = REPO_ROOT / 'config' / 'core_equivalent_cases.json'

PROFILES = ['permissive', 'moderate', 'strict', 'very_strict']
PRIMARY_PROFILE = 'moderate'

with open(CANONICAL_PATH) as f:
    canonical = json.load(f)
with open(EEE_PATH) as f:
    eee_data = json.load(f)
with open(CORE_EQUIV_PATH) as f:
    core_equiv = json.load(f)

eee_lookup = {c['case_id']: c.get('eee_feature_overlay', {}) for c in eee_data['cases']}
orig12 = sorted(canonical['cases'].keys())
core8 = core_equiv['case_ids']

print(f'Engine loaded: {engine.__name__}')
print(f'Original 12 cases: {len(orig12)}')
print(f'Core-equivalent 8 cases: {len(core8)}')

Engine loaded: corrected_public_engine_v1_1
Original 12 cases: 12
Core-equivalent 8 cases: 8


## 2.1 Feature Routing Architecture

The manuscript reports results from three distinct analytical layers, each with a different feature source. This separation is methodologically critical.

| Layer | Cases | Feature Source | Expected |
|-------|-------|---------------|----------|
| Layer 1 | 12 original | I1 canonical | 11/12 reject |
| Layer 2 | 20 failures + 12 controls | I2 base + EEE overlay | TP=20, TN=12 |
| Expanded | 12 (I1) + 49 (I2+EEE) + 30 controls (I2) | Tiered | 60/61, 30/30 |

## 2.2 Layer 1 — Primary 12-Case Analysis

Under the moderate governance profile, the framework rejected 11 of 12 historical failure cases. The sole approved case (Google Diabetic Retinopathy screening in Thailand) passed the safety gate by a margin of just 0.05 above threshold.

In [2]:
# Layer 1: I1 canonical -> I1 engine, moderate, replay_mode
layer1_results = []
for cid in orig12:
    flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
    r = engine.evaluate_case({'case_id': cid, 'features': flat},
                             profile_name=PRIMARY_PROFILE, mode=engine.MODE_REPLAY)
    layer1_results.append({'case_id': cid, **r})

rejected = [r for r in layer1_results if not r['approved']]
approved = [r for r in layer1_results if r['approved']]

print(f'Layer 1 (moderate, replay): {len(rejected)}/12 rejected, {len(approved)}/12 approved')
for r in approved:
    print(f'  Approved: {r["case_id"]}')

assert len(rejected) == 11, f'Q1: Expected 11 rejected, got {len(rejected)}'
assert len(approved) == 1 and approved[0]['case_id'] == 'google_dr'
print('\nQ1: 11/12 rejected (91.7%) ✓')

Layer 1 (moderate, replay): 11/12 rejected, 1/12 approved
  Approved: google_dr

Q1: 11/12 rejected (91.7%) ✓


### 2.3 Gate Failure Distribution

The safety gate was the binding constraint in 83% of rejected cases, consistent with its architectural role as the primary safeguard.

In [3]:
# Count gate failures across rejected cases
gate_names = ['gate_safety', 'gate_evidence', 'gate_bias', 'gate_calibration', 'gate_traceability']
gate_fail_counts = {g: 0 for g in gate_names}
total_gate_failures = 0

for r in rejected:
    for g in gate_names:
        if r.get(g, 1) == 0:  # gate failed
            gate_fail_counts[g] += 1
            total_gate_failures += 1

safety_binding = gate_fail_counts['gate_safety']
mean_failures = total_gate_failures / len(rejected)

print('Gate failure distribution (11 rejected cases):')
print(f'  Safety:       {gate_fail_counts["gate_safety"]}/12 ({gate_fail_counts["gate_safety"]/12*100:.0f}%)')
print(f'  Bias:         {gate_fail_counts["gate_bias"]}/12 ({gate_fail_counts["gate_bias"]/12*100:.0f}%)')
print(f'  Calibration:  {gate_fail_counts["gate_calibration"]}/12 ({gate_fail_counts["gate_calibration"]/12*100:.0f}%)')
print(f'  Traceability: {gate_fail_counts["gate_traceability"]}/12 ({gate_fail_counts["gate_traceability"]/12*100:.0f}%)')
print(f'  Evidence:     {gate_fail_counts["gate_evidence"]}/12 ({gate_fail_counts["gate_evidence"]/12*100:.0f}%)')
print(f'  Mean failures/rejection: {mean_failures:.3f} (manuscript: 2.5, see N7 annotation)')
print(f'  Min gate failures: {min(sum(1 for g in gate_names if r.get(g, 1) == 0) for r in rejected)}')

assert safety_binding == 10, f'Q2: Expected 10, got {safety_binding}'
assert gate_fail_counts['gate_bias'] == 6, f'Q3: Expected 6'
assert gate_fail_counts['gate_calibration'] == 5, f'Q4: Expected 5'
assert gate_fail_counts['gate_traceability'] == 5, f'Q5: Expected 5'
assert gate_fail_counts['gate_evidence'] == 3, f'Q6: Expected 3'
assert min(sum(1 for g in gate_names if r.get(g, 1) == 0) for r in rejected) >= 2, 'Q8: Min >= 2'
print('\nQ2-Q6, Q8 ✓')

# Q9: Google DR safety margin
gdr = [r for r in layer1_results if r['case_id'] == 'google_dr'][0]
gdr_safety = canonical['cases']['google_dr']['features']['intrinsic_safety']['value_primary']
margin = gdr_safety - 0.50
print(f'\nGoogle DR safety margin: {gdr_safety} - 0.50 = {margin:.2f}')
assert abs(margin - 0.05) < 1e-6, f'Q9: Expected 0.05, got {margin}'
print('Q9: Google DR margin 0.05 ✓')

Gate failure distribution (11 rejected cases):
  Safety:       10/12 (83%)
  Bias:         6/12 (50%)
  Calibration:  5/12 (42%)
  Traceability: 5/12 (42%)
  Evidence:     3/12 (25%)
  Mean failures/rejection: 2.636 (manuscript: 2.5, see N7 annotation)
  Min gate failures: 2

Q2-Q6, Q8 ✓

Google DR safety margin: 0.55 - 0.50 = 0.05
Q9: Google DR margin 0.05 ✓


### 2.4 Compensatory Comparison

Two cases exhibited divergence between the non-compensatory and compensatory frameworks: Google Flu Trends and Uber Autonomous Vehicle.

In [4]:
# Compare non-comp vs comp
divergences = []
for r in layer1_results:
    nc_approved = r['approved']
    comp_score = r.get('compensatory_score', 0)
    comp_threshold = r.get('compensatory_threshold', 0.5)
    comp_approved = comp_score >= comp_threshold if comp_score else False
    if nc_approved != comp_approved:
        divergences.append(r['case_id'])
    
print(f'Compensatory divergences: {len(divergences)} cases')
for d in divergences:
    r = [x for x in layer1_results if x['case_id'] == d][0]
    print(f'  {d}: comp_score={r.get("compensatory_score", "N/A")}')

agree_count = 12 - len(divergences)
gf = [r for r in layer1_results if r['case_id'] == 'google_flu'][0]
ua = [r for r in layer1_results if r['case_id'] == 'uber_av'][0]
gf_score = gf.get('compensatory_score', 0)
ua_score = ua.get('compensatory_score', 0)

assert len(divergences) == 2, f'Q10: Expected 2 divergences, got {len(divergences)}'
assert agree_count == 10, f'Q11: Expected 10 agreements'
assert abs(gf_score - 0.5675) < 1e-4, f'Q12: Expected ~0.5675, got {gf_score}'
assert abs(ua_score - 0.5125) < 1e-4, f'Q13: Expected ~0.5125, got {ua_score}'
print(f'\nQ10: 2 divergences ✓, Q11: 10/12 agree ✓')
print(f'Q12: Google Flu {gf_score:.4f} ✓, Q13: Uber AV {ua_score:.4f} ✓')

Compensatory divergences: 2 cases
  google_flu: comp_score=0.5675
  uber_av: comp_score=0.5125

Q10: 2 divergences ✓, Q11: 10/12 agree ✓
Q12: Google Flu 0.5675 ✓, Q13: Uber AV 0.5125 ✓


### 2.5 Multi-Profile Analysis

In [5]:
# All profiles x 12 cases
profile_results = {}
for prof in PROFILES:
    results = []
    for cid in orig12:
        flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        r = engine.evaluate_case({'case_id': cid, 'features': flat},
                                 profile_name=prof, mode=engine.MODE_REPLAY)
        results.append(r)
    n_rejected = sum(1 for r in results if not r['approved'])
    profile_results[prof] = n_rejected
    print(f'{prof:15s}: {n_rejected}/12 rejected')

assert profile_results['permissive'] == 8, f'Q18: Expected 8'
assert profile_results['strict'] == 12, f'Q19: Expected 12'
assert profile_results['very_strict'] == 12, f'Q20: Expected 12'
print('\nQ18: permissive 8/12 ✓, Q19: strict 12/12 ✓, Q20: very_strict 12/12 ✓')

permissive     : 8/12 rejected
moderate       : 11/12 rejected
strict         : 12/12 rejected
very_strict    : 12/12 rejected

Q18: permissive 8/12 ✓, Q19: strict 12/12 ✓, Q20: very_strict 12/12 ✓


## 2.6 Layer 2 — Core-Equivalent Confusion Matrix

The 20 core-equivalent failure cases (12 original + 8 upgraded) are processed with EEE-overlay-merged features, where triangulated evidence values replace base encodings.

In [6]:
# Layer 2: I2 base + EEE overlay -> I1 engine
# 20 failure cases (12 original + 8 core-equiv)
failure_ids_20 = orig12 + core8
tp = 0
fn = 0
gate_analysis = {g: {'fail_count': 0, 'ctrl_fail': 0} for g in gate_names}

for cid in failure_ids_20:
    with open(BENCHMARK_DIR / f'{cid}.json') as f:
        base = json.load(f)
    overlay = eee_lookup.get(cid, {})
    merged = merge_features(base['features'], overlay)
    r = engine.evaluate_case({'case_id': cid, 'features': merged},
                             profile_name=PRIMARY_PROFILE, mode=engine.MODE_REPLAY)
    if not r['approved']:
        tp += 1
    else:
        fn += 1
    for g in gate_names:
        if r.get(g, 1) == 0:
            gate_analysis[g]['fail_count'] += 1

# 12 controls
controls = sorted([f.stem for f in BENCHMARK_DIR.glob('control_*.json')])[:12]
tn = 0
fp = 0
for cid in controls:
    with open(BENCHMARK_DIR / f'{cid}.json') as f:
        base = json.load(f)
    r = engine.evaluate_case({'case_id': cid, 'features': base['features']},
                             profile_name=PRIMARY_PROFILE, mode=engine.MODE_REPLAY)
    if r['approved']:
        tn += 1
    else:
        fp += 1
    for g in gate_names:
        if r.get(g, 1) == 0:
            gate_analysis[g]['ctrl_fail'] += 1

print(f'Confusion Matrix: TP={tp}, FN={fn}, TN={tn}, FP={fp}')
print(f'Sensitivity: {tp/(tp+fn):.3f}')
print(f'Specificity: {tn/(tn+fp):.3f}')

assert tp == 20, f'Q15: Expected TP=20, got {tp}'
assert fn == 0, f'Q15: Expected FN=0, got {fn}'
assert tn == 12, f'Q15: Expected TN=12, got {tn}'
assert fp == 0, f'Q15: Expected FP=0, got {fp}'
print('\nQ15: TP=20, FN=0, TN=12, FP=0 ✓')
print('Q16: 12 original + 8 upgraded = 20 ✓')

# Q17: discriminative gates
print(f'\nGate analysis across 20 failures / 12 controls:')
for g in gate_names:
    print(f'  {g}: {gate_analysis[g]["fail_count"]}/20 failures fail, {gate_analysis[g]["ctrl_fail"]}/12 controls fail')
print('Q17: Discriminative gate analysis complete')

Confusion Matrix: TP=20, FN=0, TN=12, FP=0


Sensitivity: 1.000
Specificity: 1.000

Q15: TP=20, FN=0, TN=12, FP=0 ✓
Q16: 12 original + 8 upgraded = 20 ✓

Gate analysis across 20 failures / 12 controls:
  gate_safety: 19/20 failures fail, 0/12 controls fail
  gate_evidence: 18/20 failures fail, 0/12 controls fail
  gate_bias: 9/20 failures fail, 0/12 controls fail
  gate_calibration: 20/20 failures fail, 0/12 controls fail
  gate_traceability: 20/20 failures fail, 0/12 controls fail
Q17: Discriminative gate analysis complete


## 2.7 Expanded Benchmark (91 Cases)

The expanded benchmark applies tiered feature routing: the original 12 cases use their expert-triangulated encodings (where Google DR approves), while 49 additional failure cases use I2 base + EEE overlay, and 30 controls use base features.

In [7]:
# Expanded: TIERED routing
orig12_set = set(orig12)
fail_reject = 0
fail_approve = 0
fail_approve_ids = []
ctrl_approve = 0
ctrl_reject = 0

all_results = []  # for CSV output

for fpath in sorted(BENCHMARK_DIR.glob('*.json')):
    cid = fpath.stem
    with open(fpath) as f:
        base = json.load(f)
    
    is_control = cid.startswith('control_')
    
    if is_control:
        # Tier 3: controls use I2 base
        feat = {k: v['value_primary'] for k, v in base['features'].items()}
        tier = 'control_base'
    elif cid in orig12_set:
        # Tier 1: original 12 use I1 canonical
        feat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
        tier = 'canonical'
    else:
        # Tier 2: additional failures use I2 base + EEE overlay
        overlay = eee_lookup.get(cid, {})
        feat = merge_features(base['features'], overlay)
        tier = 'base_plus_eee'
    
    r = engine.evaluate_case({'case_id': cid, 'features': feat},
                             profile_name=PRIMARY_PROFILE, mode=engine.MODE_REPLAY)
    
    all_results.append({
        'layer': 'expanded', 'case_id': cid, 'profile': PRIMARY_PROFILE,
        'approved': r['approved'], 'tier': tier,
        'compensatory_score': r.get('compensatory_score', ''),
    })
    
    if is_control:
        if r['approved']:
            ctrl_approve += 1
        else:
            ctrl_reject += 1
    else:
        if not r['approved']:
            fail_reject += 1
        else:
            fail_approve += 1
            fail_approve_ids.append(cid)

print(f'Expanded benchmark:')
print(f'  Failures rejected: {fail_reject}/61')
print(f'  Failures approved: {fail_approve}/61 ({fail_approve_ids})')
print(f'  Controls approved: {ctrl_approve}/30')
print(f'  Controls rejected: {ctrl_reject}/30')

assert fail_reject == 60, f'Q34: Expected 60, got {fail_reject}'
assert ctrl_approve == 30, f'Q35: Expected 30, got {ctrl_approve}'
assert fail_approve_ids == ['google_dr'], f'Expected only google_dr approved'
print('\nQ34: 60/61 rejected ✓, Q35: 30/30 approved ✓')
print('Google DR sole approval under tiered routing ✓')

Expanded benchmark:
  Failures rejected: 60/61
  Failures approved: 1/61 (['google_dr'])
  Controls approved: 30/30
  Controls rejected: 0/30

Q34: 60/61 rejected ✓, Q35: 30/30 approved ✓
Google DR sole approval under tiered routing ✓


## 2.8 Write Outputs

In [8]:
# Write replay_results.csv (Layer 1 all profiles + expanded)
output_path = REPO_ROOT / 'outputs' / 'tables' / 'replay_results.csv'
with open(output_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['layer', 'case_id', 'profile', 'approved', 'tier', 'compensatory_score',
                'gate_safety', 'gate_evidence', 'gate_bias', 'gate_calibration', 'gate_traceability'])
    
    # Layer 1: all profiles
    for prof in PROFILES:
        for cid in orig12:
            flat = {k: v['value_primary'] for k, v in canonical['cases'][cid]['features'].items()}
            r = engine.evaluate_case({'case_id': cid, 'features': flat},
                                     profile_name=prof, mode=engine.MODE_REPLAY)
            w.writerow(['layer1', cid, prof, r['approved'], 'canonical',
                        f"{r.get('compensatory_score', ''):.6f}" if r.get('compensatory_score') is not None else '',
                        r.get('gate_safety', ''), r.get('gate_evidence', ''),
                        r.get('gate_bias', ''), r.get('gate_calibration', ''),
                        r.get('gate_traceability', '')])
    
    # Expanded
    for row in all_results:
        w.writerow(['expanded', row['case_id'], row['profile'], row['approved'], row['tier'],
                     f"{row['compensatory_score']:.6f}" if isinstance(row['compensatory_score'], float) else '',
                     '', '', '', '', ''])

print(f'Written: {output_path}')

# Write confusion_matrix.csv
cm_path = REPO_ROOT / 'outputs' / 'tables' / 'confusion_matrix.csv'
with open(cm_path, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'value'])
    w.writerow(['TP', tp])
    w.writerow(['FN', fn])
    w.writerow(['TN', tn])
    w.writerow(['FP', fp])
    w.writerow(['sensitivity', f'{tp/(tp+fn):.3f}'])
    w.writerow(['specificity', f'{tn/(tn+fp):.3f}'])
print(f'Written: {cm_path}')

# Write run log
log_path = REPO_ROOT / 'outputs' / 'logs' / 'replay_run_log.txt'
with open(log_path, 'w', newline='', encoding='utf-8') as f:
    f.write('Paper 4 Historical Replay Execution Log\n')
    f.write(f'Engine: corrected_public_engine_v1_1\n')
    f.write(f'Primary profile: {PRIMARY_PROFILE}\n')
    f.write(f'Layer 1: {len(orig12)} cases, {profile_results[PRIMARY_PROFILE]}/12 rejected\n')
    f.write(f'Layer 2: TP={tp}, FN={fn}, TN={tn}, FP={fp}\n')
    f.write(f'Expanded: {fail_reject}/61 failures rejected, {ctrl_approve}/30 controls approved\n')
print(f'Written: {log_path}')

Written: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-4-historical-replay\outputs\tables\replay_results.csv
Written: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-4-historical-replay\outputs\tables\confusion_matrix.csv
Written: C:\Users\Walter.Brown\EAA_Portfolio\ethical-alpha-audit-paper-4-historical-replay\outputs\logs\replay_run_log.txt
